# SageMaker Pipeline for Hugging Face Model Fine-tuning

This notebook demonstrates how to:
- Fine-tune a Hugging Face model with data from S3
- Orchestrate the process with a SageMaker Pipeline
- Register the trained model in the SageMaker Model Registry
- Prepare it for deployment to a SageMaker inference endpoint

⚠️ **Note:** This is a skeleton pipeline to get you started. For real LLMs, use PEFT/LoRA and distributed training.

In [ ]:
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker.workflow.parameters import ParameterString, ParameterInteger
from sagemaker.workflow.steps import TrainingStep, RegisterModel
from sagemaker.workflow.pipeline import Pipeline
import boto3

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sagemaker_session = sagemaker.session.Session()
bucket = sagemaker_session.default_bucket()
print("Using bucket:", bucket)

## Define pipeline parameters

In [ ]:
model_id = ParameterString(name="ModelId", default_value="distilbert-base-uncased")
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
instance_count = ParameterInteger(name="TrainingInstanceCount", default_value=1)
training_data = ParameterString(name="TrainingData", default_value=f"s3://{bucket}/hf-dataset/train/")

## Define HuggingFace Estimator

In [ ]:
huggingface_estimator = HuggingFace(
    entry_point="train.py",
    source_dir="hf_training_script",
    instance_type=instance_type,
    instance_count=instance_count,
    role=role,
    transformers_version="4.36",
    pytorch_version="2.1",
    py_version="py310",
    hyperparameters={
        "epochs": 1,
        "train_batch_size": 32,
        "model_id": model_id
    }
)

## Create Training Step

In [ ]:
train_step = TrainingStep(
    name="HuggingFaceTraining",
    estimator=huggingface_estimator,
    inputs={"train": training_data}
)

## Register Model Step

In [ ]:
register_step = RegisterModel(
    name="RegisterHuggingFaceModel",
    estimator=huggingface_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name="HuggingFaceModelPackageGroup"
)

## Build and Submit Pipeline

In [ ]:
pipeline = Pipeline(
    name="HuggingFaceFineTunePipeline",
    parameters=[model_id, instance_type, instance_count, training_data],
    steps=[train_step, register_step],
    sagemaker_session=sagemaker_session
)

pipeline.upsert(role_arn=role)
execution = pipeline.start()
execution.wait()